This file is the version 1.0.0.1.0 for the project

In [3]:
# !git clone https://github.com/Phantom-fs/Soil-Classification-Dataset

# Load and split the images

In [1]:
import tensorflow as tf

In [4]:
dataset_dir = "Soil-Classification-Dataset/CyAUG-Dataset"
IMG_SIZE = 224
BATCH_SIZE = 32

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

Found 5096 files belonging to 7 classes.
Using 4077 files for training.


In [6]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

Found 5096 files belonging to 7 classes.
Using 1019 files for validation.


In [7]:
class_names = train_ds.class_names
print(class_names)

['Alluvial_Soil', 'Arid_Soil', 'Black_Soil', 'Laterite_Soil', 'Mountain_Soil', 'Red_Soil', 'Yellow_Soil']


In [8]:
NUM_CLASSES = len(class_names)
print(f"Number of classes: {NUM_CLASSES}")

Number of classes: 7


# Adding prefetching (Optimal Performance)

In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# Building the EfficientNetV2 Model

In [12]:
from tensorflow.keras import layers, models

In [13]:
base_model = tf.keras.applications.EfficientNetV2B0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)

base_model.trainable = False

24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 21s 1us/step


In [17]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Lambda(tf.keras.applications.efficientnet_v2.preprocess_input),

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [18]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 7, 7, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,254,167 (23.86 MB)

 Trainable params: 332,295 (1.27 MB)

 Non-trainable params: 5,921,872 (22.59 MB)

In [19]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 63s 376ms/step - accuracy: 0.8759 - loss: 0.3933 - val_accuracy: 0.9480 - val_loss: 0.1777
Epoch 2/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 42s 329ms/step - accuracy: 0.9546 - loss: 0.1360 - val_accuracy: 0.9627 - val_loss: 0.1177
Epoch 3/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 48s 372ms/step - accuracy: 0.9639 - loss: 0.1205 - val_accuracy: 0.9647 - val_loss: 0.1026
Epoch 4/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 50s 389ms/step - accuracy: 0.9755 - loss: 0.0780 - val_accuracy: 0.9696 - val_loss: 0.1072
Epoch 5/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 43s 332ms/step - accuracy: 0.9794 - loss: 0.0595 - val_accuracy: 0.9696 - val_loss: 0.1164
Epoch 6/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 46s 357ms/step - accuracy: 0.9789 - loss: 0.0603 - val_accuracy: 0.9666 - val_loss: 0.1319
Epoch 7/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 61s 480ms/step - accuracy: 0.9818 - loss: 0.0538 - val_accuracy: 0.9676 - val_loss: 0.1369
Epoch 8/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 81s 634ms/step - accuracy: 0.9826 - loss: 0

# Fine-Tuning for Better Accuracy

In [20]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 300s 2s/step - accuracy: 0.7481 - loss: 1.2107 - val_accuracy: 0.8283 - val_loss: 0.6799
Epoch 2/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 221s 2s/step - accuracy: 0.8305 - loss: 0.6380 - val_accuracy: 0.8960 - val_loss: 0.3913
Epoch 3/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 218s 2s/step - accuracy: 0.8685 - loss: 0.4814 - val_accuracy: 0.9156 - val_loss: 0.3130
Epoch 4/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 231s 2s/step - accuracy: 0.8938 - loss: 0.3904 - val_accuracy: 0.9264 - val_loss: 0.2735
Epoch 5/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 303s 2s/step - accuracy: 0.9058 - loss: 0.3356 - val_accuracy: 0.9411 - val_loss: 0.2544
Epoch 6/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 353s 3s/step - accuracy: 0.9225 - loss: 0.2772 - val_accuracy: 0.9421 - val_loss: 0.2282
Epoch 7/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 336s 3s/step - accuracy: 0.9267 - loss: 0.2521 - val_accuracy: 0.9470 - val_loss: 0.2117
Epoch 8/10
128/128 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.9345 - loss: 0.2257 - val_accu

# Save the Trained Model

In [21]:
model.save("soil_classification_model_V_1.0.0.1.0.keras")

# Testing the Model

In [23]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "soil_classification_model_V_1.0.0.1.0.keras",
    custom_objects={
        "preprocess_input": tf.keras.applications.efficientnet_v2.preprocess_input
    }
)

## Image Prediction Code

In [24]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

In [25]:
IMG_SIZE = 224

In [26]:
def predict_image(img_path, model, class_names):
    # Load & resize image
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)

    # Add batch dimension
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(img_array)

    # Extract results
    predicted_index = np.argmax(predictions[0])
    predicted_class = class_names[predicted_index]
    confidence = float(np.max(predictions[0]))

    print(f"\nPrediction: {predicted_class}")
    print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")

    return predicted_class, confidence

In [28]:
class_names = ['Alluvial Soil', 'Black Soil', 'Laterite Soil',
               'Red Soil', 'Yellow Soil', 'Mountain Soil', 'Arid Soil']

predict_image("testImage00.jpg", model, class_names)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

Prediction: Red Soil
Confidence: 0.9379 (93.79%)


('Red Soil', 0.937882661819458)

# Upgraded Prediction System (Multi-Image Soft Voting)

In [29]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import os

IMG_SIZE = 224


def predict_images(img_dir, model, class_names):
    if not os.path.isdir(img_dir):
        print(f"Directory not found: {img_dir}")
        return None, None

    # Collect all image files (common extensions)
    valid_exts = (".jpg", ".jpeg", ".png", ".bmp")
    img_paths = [
        os.path.join(img_dir, f)
        for f in os.listdir(img_dir)
        if f.lower().endswith(valid_exts)
    ]

    if not img_paths:
        print("No images provided.")
        return None, None

    all_predictions = []

    print("\nIndividual Predictions:")
    print("-" * 40)

    for img_path in img_paths:
        img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        img_array = image.img_to_array(img)

        # Preprocess BEFORE batching (important)
        img_array = tf.keras.applications.efficientnet_v2.preprocess_input(img_array)

        img_array = np.expand_dims(img_array, axis=0)

        predictions = model.predict(img_array, verbose=0)[0]

        predicted_index = np.argmax(predictions)
        predicted_class = class_names[predicted_index]
        confidence = float(np.max(predictions))

        print(f"{img_path} → {predicted_class} ({confidence:.2%})")

        all_predictions.append(predictions)

    # ✅ Soft Voting (Average Probabilities)
    mean_predictions = np.mean(all_predictions, axis=0)

    final_index = np.argmax(mean_predictions)
    final_class = class_names[final_index]
    final_confidence = float(np.max(mean_predictions))

    print("\nFinal Ensemble Decision:")
    print("-" * 40)
    print(f"Prediction: {final_class}")
    print(f"Confidence: {final_confidence:.4f} ({final_confidence:.2%})")

    return final_class, final_confidence

In [32]:
predict_images("prediction_data", model, class_names)


Individual Predictions:
----------------------------------------
prediction_data\17.jpg → Alluvial Soil (86.38%)
prediction_data\18.jpg → Arid Soil (98.08%)
prediction_data\19.jpg → Alluvial Soil (98.14%)
prediction_data\20.jpg → Alluvial Soil (92.74%)
prediction_data\21.jpg → Red Soil (90.70%)
prediction_data\22.jpg → Alluvial Soil (70.83%)
prediction_data\23.jpg → Alluvial Soil (92.34%)
prediction_data\24.jpg → Red Soil (93.75%)
prediction_data\25.jpg → Alluvial Soil (99.98%)
prediction_data\testImage00.jpg → Red Soil (93.79%)
prediction_data\testImage01.jpg → Laterite Soil (43.13%)
prediction_data\testImage02.jpg → Laterite Soil (75.84%)

Final Ensemble Decision:
----------------------------------------
Prediction: Alluvial Soil
Confidence: 0.4762 (47.62%)


('Alluvial Soil', 0.47620508074760437)

In [35]:
base_model = model.get_layer("efficientnetv2-b0")

for layer in base_model.layers[-20:]:
    print(layer.name)

block6g_drop
block6g_add
block6h_expand_conv
block6h_expand_bn
block6h_expand_activation
block6h_dwconv2
block6h_bn
block6h_activation
block6h_se_squeeze
block6h_se_reshape
block6h_se_reduce
block6h_se_expand
block6h_se_excite
block6h_project_conv
block6h_project_bn
block6h_drop
block6h_add
top_conv
top_bn
top_activation
